# Step 5b (Kaggle): Fairness Audit of Retrieval (RQ1)

**Inputs**
- `retrieval_results.json` (output of Step 5a) - add it as a Kaggle input
  (upload as a dataset, or 'Add Input' -> Notebook Output of your 5a run).
- Institution labels (Step 1b, Raj) - auto-downloaded via kagglehub from
  `rajlucka/fairsearch-qbio-institution-labels`. You do NOT upload labels.

**What changed vs the local version**
- Kaggle path detection (like the Step 5a Kaggle notebook).
- If queries carry a `type` field, the MAIN SPD/SRR is computed on `neutral`
  only (RQ1/RQ3); `contradictory` is held out for RQ2.
- A coverage + grouped breakdown table is printed to inform the Step 1b redo.

**Metrics**
- SPD = retrieved_elite_share - corpus_elite_share (0 = parity; >0 = elite over-retrieved)
- SRR = (retrieved_elite_share / corpus_elite_share) / (retrieved_other_share / corpus_other_share) (ideal = 1.0)
- Bootstrap 95% CI (resampling over queries) + one-sample binomial test.

In [ ]:
import json, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(42)  # fixed seed for reproducible bootstrap

In [ ]:
# Cell 3 - Detect environment and set paths (Kaggle-adapted)
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
print(f"Environment: {'Kaggle' if ON_KAGGLE else 'Local'}")

if ON_KAGGLE:
    # retrieval_results.json = output of Step 5a.
    # Add it to this notebook as an input: either upload it as a dataset,
    # or use 'Add Input' -> Notebook Output of your Step 5a run.
    _cands = [
        Path("/kaggle/input/fairsearch-qbio-retrieval-results/retrieval_results.json"),
        Path("/kaggle/input/datasets/yanbochen928/fairsearch-qbio-retrieval-results/retrieval_results.json"),
    ]
    _found = [p for p in _cands if p.exists()]
    if not _found:
        _found = list(Path("/kaggle/input").rglob("retrieval_results.json"))
    if not _found:
        raise FileNotFoundError("retrieval_results.json not found under /kaggle/input. Add it as an input.")
    RETRIEVAL_PATH = str(_found[0])
    OUT_DIR = "/kaggle/working"
else:
    RETRIEVAL_PATH = str(next(p for p in [
        Path("../data/processed/retrieval_results.json"),
        Path("data/processed/retrieval_results.json"),
    ] if p.exists()))
    OUT_DIR = str(Path(RETRIEVAL_PATH).parent)

# Step 1b labels (Raj) - auto-downloaded via kagglehub in the next-but-one cell.
# You do NOT upload the label file yourself.
KAGGLE_DATASET = "rajlucka/fairsearch-qbio-institution-labels"
LABEL_FILE = "qbio_institution_labels.json"

CONFIG = {
    "paper_id_col": "paper_id",
    "is_elite_col": "elite_label",  # 1.0 -> elite, 0.0 -> other, NaN -> unlabeled
    "region_col":   "region",       # ISO country code
}
K = 10

print("RETRIEVAL_PATH:", RETRIEVAL_PATH)
print("OUT_DIR       :", OUT_DIR)

In [ ]:
#load data

with open(RETRIEVAL_PATH) as f:
    retrieval = json.load(f)
print(f'Loaded {len(retrieval)} queries from {RETRIEVAL_PATH}')
print('Keys:', list(retrieval[0].keys()))
assert all(len(q['retrieved_paper_ids']) == K for q in retrieval), 'expected top-K per query'

In [ ]:
import kagglehub
kh_path = kagglehub.dataset_download(KAGGLE_DATASET)
labels_df = pd.read_json(os.path.join(kh_path, LABEL_FILE))
print(f'Loaded {len(labels_df):,} label rows | columns: {list(labels_df.columns)}')
labels_df.head(10)

In [ ]:
# normalize labels

def build_group_map(df, cfg):
    pid, col = cfg['paper_id_col'], cfg['is_elite_col']
    g = {}
    for _, row in df.iterrows():
        val = row.get(col)
        # NaN -> unlabeled (no affiliation found); otherwise 1=elite, 0=other
        if pd.isna(val):
            grp = 'unlabeled'
        else:
            grp = 'elite' if float(val) == 1.0 else 'other'
        g[str(row[pid])] = grp
    return g

group_map = build_group_map(labels_df, CONFIG)
from collections import Counter
corpus_counts = Counter(group_map.values())
print('Corpus label distribution:', dict(corpus_counts))

# Corpus reference rates over the LABELED subset (elite + other)
n_labeled_corpus = corpus_counts['elite'] + corpus_counts['other']
corpus_elite_share = corpus_counts['elite'] / n_labeled_corpus
corpus_other_share = corpus_counts['other'] / n_labeled_corpus
print(f'Labeled corpus = {n_labeled_corpus:,} | elite base rate = {corpus_elite_share:.3f} '
      f'(expected 0.233 per Step 1b)')

In [ ]:
#coverage check

rows = []
for q in retrieval:
    for rank, pid in enumerate(q['retrieved_paper_ids']):
        grp = group_map.get(str(pid), 'unlabeled')
        rel = q['relevance'][rank] if 'relevance' in q else None
        rows.append({'query_id': q['query_id'], 'subcategory': q['subcategory'], 'type': q.get('type','neutral'),
                     'rank': rank, 'paper_id': pid, 'group': grp, 'relevance': rel})
slots = pd.DataFrame(rows)

n_total   = len(slots)
n_elite   = (slots.group == 'elite').sum()
n_other   = (slots.group == 'other').sum()
n_labeled = n_elite + n_other
n_unlab   = (slots.group == 'unlabeled').sum()

print(f'Total retrieval slots : {n_total}')
print(f'Labeled (elite+other) : {n_labeled}  ({n_labeled/n_total:.1%})')
print(f'  - elite             : {n_elite}')
print(f'  - other             : {n_other}')
print(f'Unlabeled (excluded)  : {n_unlab}  ({n_unlab/n_total:.1%})')
print()

In [ ]:
# Restrict MAIN analysis to neutral queries (RQ1/RQ3); keep full set for coverage.
slots_full = slots.copy()
_types = set(slots["type"].dropna().unique()) if "type" in slots.columns else set()
if _types - {"neutral"}:
    slots = slots[slots["type"] == "neutral"].copy()
    print(f"Main analysis = neutral only: {len(slots)} of {len(slots_full)} slots "
          f"(contradictory held out for RQ2).")
else:
    print(f"No contradictory queries detected; main analysis uses all {len(slots)} slots.")

In [ ]:
# Coverage + grouped breakdown (ties to the Step 1b go/no-go decision)
def _cov(df):
    n = len(df)
    lab = df[df.group.isin(["elite", "other"])]
    nl = len(lab); ne = int((lab.group == "elite").sum())
    return {"n_slots": n, "n_labeled": nl,
            "coverage_%": round(100 * nl / n, 1) if n else 0.0,
            "n_elite": ne,
            "elite_share": round(ne / nl, 3) if nl else float("nan")}

def _qnum(qid):
    try:
        return int(str(qid)[1:])
    except Exception:
        return -1

groups = {"all": slots_full}
if "type" in slots_full.columns:
    sf = slots_full.copy()
    sf["qnum"] = sf["query_id"].map(_qnum)
    groups["original_neutral q001-050"] = sf[(sf["type"] == "neutral") & (sf["qnum"] <= 50)]
    groups["expanded_neutral q051-100"] = sf[(sf["type"] == "neutral") & (sf["qnum"].between(51, 100))]
    groups["all_neutral"]               = sf[sf["type"] == "neutral"]
    groups["contradictory q101-150"]    = sf[sf["type"] == "contradictory"]

cov_table = pd.DataFrame({k: _cov(v) for k, v in groups.items() if len(v)}).T
print(cov_table)
print("")
print("Read SPD/SRR (neutral) together with coverage_% to decide if Step 1b needs a redo.")

In [ ]:
# SPD & SRR

lab = slots[slots.group.isin(['elite', 'other'])]
e = (lab.group == 'elite').sum()
n = len(lab)
retrieved_elite_share = e / n
retrieved_other_share = 1 - retrieved_elite_share

SPD = retrieved_elite_share - corpus_elite_share
# SRR: over-representation lift of elite vs other; ideal = 1.0
SRR = (retrieved_elite_share / corpus_elite_share) / (retrieved_other_share / corpus_other_share)

print(f'Retrieved elite share : {retrieved_elite_share:.3f}  ({e}/{n})')
print(f'Corpus elite share    : {corpus_elite_share:.3f}')
print(f'SPD                    : {SPD:+.3f}   (>0 => elite over-retrieved)')
print(f'SRR                    : {SRR:.3f}')

In [ ]:
# uncertainty: bootstrap CI (resample queries) + binomial test

def pooled_metrics(query_subset):
    sub = slots[slots.query_id.isin(query_subset) & slots.group.isin(['elite','other'])]
    if len(sub) == 0:
        return np.nan, np.nan
    es = (sub.group == 'elite').mean()
    spd = es - corpus_elite_share
    srr = (es / corpus_elite_share) / ((1-es) / corpus_other_share) if 0 < es < 1 else np.nan
    return spd, srr

qids = [q['query_id'] for q in retrieval if q.get('type','neutral')=='neutral']
B = 5000
boot_spd, boot_srr = [], []
for _ in range(B):
    sample = rng.choice(qids, size=len(qids), replace=True)
    spd, srr = pooled_metrics(sample)
    boot_spd.append(spd); boot_srr.append(srr)
boot_spd = np.array(boot_spd); boot_srr = np.array(boot_srr)

spd_ci = np.nanpercentile(boot_spd, [2.5, 97.5])
srr_ci = np.nanpercentile(boot_srr, [2.5, 97.5])
print(f'SPD 95% CI: [{spd_ci[0]:+.3f}, {spd_ci[1]:+.3f}]')
print(f'SRR 95% CI: [{srr_ci[0]:.3f}, {srr_ci[1]:.3f}]')

# One-sample binomial test: is retrieved elite count different from base rate?
bt = stats.binomtest(int(e), int(n), p=corpus_elite_share, alternative='two-sided')
print(f'Binomial test p-value (H0: elite share == {corpus_elite_share:.3f}): {bt.pvalue:.4f}')
sig = 'SIGNIFICANT' if bt.pvalue < 0.05 else 'not significant'
print(f'-> {sig} at alpha=0.05')

In [ ]:
rel = slots[(slots.group.isin(['elite','other'])) & (slots.relevance == 1)]
if len(rel) >= 10:
    e_r = (rel.group == 'elite').sum(); n_r = len(rel)
    es_r = e_r / n_r
    spd_r = es_r - corpus_elite_share
    srr_r = (es_r/corpus_elite_share) / ((1-es_r)/corpus_other_share) if 0 < es_r < 1 else np.nan
    print(f'Relevant labeled slots: {n_r} (elite={e_r})')
    print(f'SPD (relevant only): {spd_r:+.3f} | SRR: {srr_r:.3f}')

In [ ]:
if CONFIG.get('region_col') and CONFIG['region_col'] in labels_df.columns:
    region_map = {str(r[CONFIG['paper_id_col']]): r[CONFIG['region_col']]
                  for _, r in labels_df.iterrows()}
    slots['region'] = slots.paper_id.map(lambda p: region_map.get(str(p)))
    reg = slots[slots.group.isin(['elite','other'])].dropna(subset=['region'])
    print('Retrieved labeled papers by region:')
    print(reg.region.value_counts())
    print()
    print('Elite share within each region (descriptive):')
    print(reg.groupby('region').apply(lambda d: (d.group=='elite').mean()).round(3))

In [ ]:
def subcat_row(d):
    lab_d = d[d.group.isin(['elite','other'])]
    n_l = len(lab_d); e_l = (lab_d.group=='elite').sum()
    es = e_l/n_l if n_l else np.nan
    return pd.Series({'labeled': n_l, 'elite': e_l,
                      'elite_share': round(es,3) if n_l else np.nan})
by_sub = slots.groupby('subcategory').apply(subcat_row).sort_values('labeled', ascending=False)
print(by_sub)
print('\nNote: most subcategories have very few labeled slots; read directionally only.')

In [ ]:
summary = {
    'k': K,
    'n_queries': len(retrieval),
    'n_slots_total': int(n_total),
    'n_labeled': int(n_labeled),
    'n_elite': int(n_elite),
    'n_other': int(n_other),
    'n_unlabeled': int(n_unlab),
    'corpus_elite_share': round(float(corpus_elite_share), 4),
    'retrieved_elite_share': round(float(retrieved_elite_share), 4),
    'SPD': round(float(SPD), 4),
    'SRR': round(float(SRR), 4),
    'SPD_95CI': [round(float(spd_ci[0]),4), round(float(spd_ci[1]),4)],
    'SRR_95CI': [round(float(srr_ci[0]),4), round(float(srr_ci[1]),4)],
    'binomial_p': round(float(bt.pvalue), 4),
}
per_query = []
for q in retrieval:
    sub = slots[slots.query_id == q['query_id']]
    labq = sub[sub.group.isin(['elite','other'])]
    per_query.append({
        'query_id': q['query_id'],
        'subcategory': q['subcategory'],
        'n_labeled': int(len(labq)),
        'n_elite': int((labq.group=='elite').sum()),
        'n_other': int((labq.group=='other').sum()),
    })
out = {'summary': summary, 'per_query': per_query}
os.makedirs(OUT_DIR, exist_ok=True)
out_path = os.path.join(OUT_DIR, 'fairness_results.json')
with open(out_path, 'w') as f:
    json.dump(out, f, indent=2)
print('saved to', out_path)
print(json.dumps(summary, indent=2))

In [ ]:
#plot

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(['Corpus\n(base rate)', 'Retrieved\n(top-%d)' % K],
              [corpus_elite_share, retrieved_elite_share],
              color=['#888', '#c0392b'])
ax.axhline(corpus_elite_share, ls='--', c='#888', lw=1)
ax.set_ylabel('Elite-institution share')
ax.set_title(f'Elite representation  (SPD={SPD:+.3f}, SRR={SRR:.2f})\n'
             f'n_labeled={n_labeled}, binomial p={bt.pvalue:.3f}')
for b, v in zip(bars, [corpus_elite_share, retrieved_elite_share]):
    ax.text(b.get_x()+b.get_width()/2, v+0.005, f'{v:.1%}', ha='center')
plt.tight_layout()
plt.savefig('step5b_elite_share.png', dpi=120)
plt.show()